In [ ]:
# Purpose - plots cutaways for the big boy (141)

findex = 140

step_choice = "last"

In [ ]:
# Imports and Folder Selection

from graphet import Data
from graphet.plugins import TristanV2 
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')
import sys
sys.path.append('../v2_proc_files')
from useful_funcs import *
from TristanRun import TristanRun

plt.close('all')

%matplotlib widget

set_standard_plot_params()

# Read Input File Data

run = TristanRun(findex)
gyrofreq = run.gyrofreq

cc = run.cc
B0 = run.B0

tstep_choices = {"first": [run.steps[0]], "last": [run.steps[-1]], "all": run.steps}

my_steps = tstep_choices[step_choice]
boilerplate = run.boilerplate + f" tstep = {my_steps[0]}"

In [ ]:
# Configure Data

d = run.config_data(my_steps, "fields", includes_pressure = False)

In [ ]:
# Choose Regions of Interest

xlim = run.xlim
ylim = run.ylim

#rect = [xlim / 2 - 4, xlim / 2 + 4, ylim / 2 - 10, ylim / 2 + 10]
#rect = [0, xlim, 0, ylim]
#rect = [50, 70, 130, 190]
rect = [0, xlim, 0, ylim]
rect = [5, xlim - 5, ylim / 2 - 240, ylim / 2 + 240]

tchosen = 0
roi = fq_cut(d.fields, tchosen, rect)

#xcut_y = 164.4
xcut_y = ylim / 10 #5950
ycut_x = xlim / 2

xstrip = fq_xcut(roi, xcut_y)
ystrip = fq_ycut(roi, ycut_x)

In [ ]:
roi = fq_cut(d.fields, 0, [0, run.xlim, 0, run.ylim])

#roi = fq_cut(d.fields, 0, [55, 65, 110, 130])

# Create a subplot with one plot
fig, ax = plt.subplots()

#xa, ya = [0.02, 0.99]

#ax.text(xa, ya, f"$\sigma$ = {run.sigma}", color='black', ha='left', va='top',
#        transform=ax.transAxes, fontsize = 2, weight = "bold",
#        bbox=dict(boxstyle='square,pad=0.3', edgecolor='white', facecolor='white', alpha=0.8))

#plt.title(f"{run.boilerplate}")
mesh = plot_basic(ax, run, roi, sdens = 0.25, cutoff = 5)

buffer = 5
yextent = 300 
plt.xlim(buffer, run.xlim - buffer)
plt.ylim(0, run.ylim)
#rect_patch = plt.Rectangle((rect[0], rect[2]), rect[1] - rect[0], rect[3] - rect[2], 
#                           linewidth=2, edgecolor='red', facecolor='none', linestyle=':')
#ax.add_patch(rect_patch)
cbar = fig.colorbar(mesh, ax=ax)
cbar.set_label("$n / n_0$", rotation=270)
plt.tight_layout()

# Show the plot
plt.show()

In [ ]:
from matplotlib.gridspec import GridSpec
from matplotlib.patches import ConnectionPatch
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

In [ ]:
lazy = 5
buffer = 10

def get_crec(width, height, center = [run.xlim / 2, run.ylim / 2]):
    x1 = center[0] - width / 2
    x2 = center[0] + width / 2
    y1 = center[1] - height / 2
    y2 = center[1] + height / 2
    return [x1, x2, y1, y2]


rects = [get_crec(240, 480), [run.xlim / 2 - 200, run.xlim / 2 + 200, 1300, 1900]]
colors = ['red', 'yellow']
titles = ['Zoom 1', 'Zoom 2']

def plot_main(ax):
    roi = fq_cut(d.fields, 0, [0 + buffer, run.xlim - buffer, 0, run.ylim])

    im = plot_fq3(roi, roi.dens, plot = ax, vabsmax = 5, positive = True, downsample = 1)
    flh = FieldLineHelper(run, 20, contour_ds = 10)
    flh.plot_field_lines(roi, 0, plot = ax, arrow_spacing = -1, linewidth = 0.1) #1000

    ax.set_xlim(buffer, run.xlim - buffer)

    #ax.set_title("Main Image")
    return im

def plot_zoom(ax, rect, color, title):
    [x1, x2, y1, y2] = rect
    roi = fq_cut(d.fields, 0, rect)
    plot_fq3(roi, roi.dens, plot = ax, vabsmax = 5, positive = True)
    flh = FieldLineHelper(run, 8, contour_ds = 1)
    flh.plot_field_lines(roi, 0, plot = ax, arrow_spacing = 200)

    ax.set_xlim(x1, x2)
    ax.set_ylim(y1, y2)
    #ax.set_title(title)
    # Show axis ticks and labels
    ax.tick_params(axis='both', which='both', direction='in', length=4)
    ax.set_xlabel("$x$ [$c/\\omega_{p}$]")
    ax.set_ylabel("$y$ [$c/\\omega_{p}$]")
    rect_patch = plt.Rectangle((x1, y1), x2 - x1, y2 - y1, edgecolor=color, facecolor='none', lw=1.5)
    return rect_patch

def connect_rect(fig, ax_main, ax_zoom, rect, color):
    [x1, x2, y1, y2] = rect
    con1 = ConnectionPatch(xyA=(x2, y1), coordsA=ax_main.transData,
                           xyB=(0, 0), coordsB=ax_zoom.transAxes,
                           color=color)
    con2 = ConnectionPatch(xyA=(x2, y2), coordsA=ax_main.transData,
                           xyB=(0, 1), coordsB=ax_zoom.transAxes,
                           color=color)
    fig.add_artist(con1)
    fig.add_artist(con2)

fig = plt.figure(figsize=(12, 18))
gs = GridSpec(2, 2, width_ratios=[3, 1.5], height_ratios=[1, 1], wspace=0.15, hspace=0.3)

ax_main = fig.add_subplot(gs[:, 0])
im = plot_main(ax_main)

plt.tight_layout(rect=[0, 0.08, 1, 1])

width_frac = 0.5
left_frac = (1 - width_frac) / 2 
left_frac = 0.32
cax = fig.add_axes([left_frac, 0.04, width_frac, 0.02])
cbar = fig.colorbar(im, cax=cax, orientation='horizontal')

# Put ticks and label on top
cbar.ax.xaxis.set_label_position('top')
cbar.ax.xaxis.tick_top()
cbar.set_label(r"$n / n_0$")

zoom_axes = []
for i, (rect, color, title) in enumerate(zip(rects, colors, titles)):
    ax_zoom = fig.add_subplot(gs[i, 1])
    rect_patch = plot_zoom(ax_zoom, rect, color, title)
    ax_main.add_patch(rect_patch)
    connect_rect(fig, ax_main, ax_zoom, rect, color)
    zoom_axes.append(ax_zoom)



save_polished_fig(name = "bigrun_zooms", dpi = 400)

plt.show()